In [2]:
from utils.metrics import mse_loss, dice_loss, precision_recall_f1, pixel_accuracy, point_matching, visualize_batch_matching  

In [3]:
# Export predictions from the updated simulation-conditioned model before evaluation.
import os
import cv2
import torch
from torch.utils.data import DataLoader
from expcolortoseed_dataset import MyDataset
from models.UNet_Resblocks import UNet_Resblocks
from utils.config import (
    CKPT_PATH_SIMULATION_CONDITIONED_LR5E4_LAST,
    OUTPUT_DIR_EXPTOSEED_COLOR_SAVED,
    EXP_FOLDER_TEST, SEED_FOLDER_EXPTOSIM_TEST_32X32,
    EXP_FOLDER_EXPTOSIM_TEST_FIXED, SEED_FOLDER_EXPTOSIM_TEST_FIXED_32X32,
)

# Match Figure 5E's last.ckpt; freeze the VAE sampling seed for reproducibility.
import random
import numpy as np
INFERENCE_RANDOM_SEED = 42
random.seed(INFERENCE_RANDOM_SEED)
np.random.seed(INFERENCE_RANDOM_SEED)
torch.manual_seed(INFERENCE_RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(INFERENCE_RANDOM_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

output_dir_exptoseed_color = OUTPUT_DIR_EXPTOSEED_COLOR_SAVED
os.makedirs(output_dir_exptoseed_color, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet_Resblocks.load_from_checkpoint(
    CKPT_PATH_SIMULATION_CONDITIONED_LR5E4_LAST, map_location=device,
).eval().to(device)

# Export both random and fixed experimental test sets with the established threshold.
inference_stems = []
for source_folder, target_folder in (
    (EXP_FOLDER_TEST, SEED_FOLDER_EXPTOSIM_TEST_32X32),
    (EXP_FOLDER_EXPTOSIM_TEST_FIXED, SEED_FOLDER_EXPTOSIM_TEST_FIXED_32X32),
):
    dataset = MyDataset(source_folder=source_folder, target_folder=target_folder, use_vae=True)
    loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)
    with torch.inference_mode():
        for batch in loader:
            source = batch['source'].to(device)
            prediction = torch.sigmoid(model(model.encode_source(source)))
            prediction = ((prediction > 0.6).to(torch.uint8) * 255).cpu().numpy()
            for index, stem in enumerate(batch['stem']):
                inference_stems.append(stem)
                destination = os.path.join(output_dir_exptoseed_color, f'{stem}.png')
                if not cv2.imwrite(str(destination), prediction[index, 0]):
                    raise OSError(f'Could not save prediction: {destination}')

assert len(inference_stems) == 132 and len(set(inference_stems)) == 132
assert sum(stem.startswith('Fixed_') for stem in inference_stems) == 36
import hashlib
checkpoint_path = CKPT_PATH_SIMULATION_CONDITIONED_LR5E4_LAST
checkpoint_hash = hashlib.sha256()
with open(checkpoint_path, 'rb') as checkpoint_file:
    for block in iter(lambda: checkpoint_file.read(4 * 1024 * 1024), b''):
        checkpoint_hash.update(block)
evaluation_provenance = {
    'checkpoint': checkpoint_path,
    'checkpoint_selection': 'last.ckpt',
    'checkpoint_sha256': checkpoint_hash.hexdigest(),
    'random_seed': INFERENCE_RANDOM_SEED,
    'batch_size': 16,
    'probability_threshold': 0.6,
    'matching_algorithm': 'hungarian',
    'tolerance_radius_pixels': 4,
    'n_images': 132, 'n_primary': 96, 'n_fixed': 36,
    'vae_encoding': 'existing encode_img, sampled latent; unchanged',
    'source_preprocessing': 'existing backgroundwhite, 25 px crop',
    'inference_device': str(device),
}
del model
print(f'Saved updated-model predictions to {output_dir_exptoseed_color}')

if torch.cuda.is_available():
    torch.cuda.empty_cache()


/hpc/dctrl/ks723/miniconda3/envs/pytorch_PA_patternprediction/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-11-25 07:20:31,823] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/hpc/dctrl/ks723/miniconda3/envs/pytorch_PA_patternprediction/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved updated-model predictions to /hpc/dctrl/ks723/Physics_constrained_DL_pattern_prediction/information_encoding_decoding/inference/EXPTOSEED_COLOR_v2


In [4]:
# Now checking for experiments 

# Check with images from Experiment to seed matching, using vae 

from utils.config import OUTPUT_DIR_EXPTOSEED_COLOR_SAVED, SEED_FOLDER_EXPTOSIM_TEST_FIXED_NONFIXED_32X32
import os 
import torch
import cv2
import numpy as np
from utils.preprocess import preprocess_seed

prediction_files = sorted(name for name in os.listdir(OUTPUT_DIR_EXPTOSEED_COLOR_SAVED)
                          if name.lower().endswith('.png'))
gt_files = sorted(name for name in os.listdir(SEED_FOLDER_EXPTOSIM_TEST_FIXED_NONFIXED_32X32)
                  if name.lower().endswith('.png'))
assert len(prediction_files) == len(gt_files) == 132
assert [os.path.splitext(name)[0] for name in prediction_files] == [os.path.splitext(name)[0] for name in gt_files], 'Prediction/target stems differ'



# prediction files has output prefix, and gt files has input prefix, apart from that they match

pred_img_tensor_list= []
gt_img_tensor_list= []

mse_loss_list= []
dice_loss_list= []
precision_recall_f1_list =[]
pixel_accuracy_list =[]

precision_rtol_list= []
accuracy_rtol_list= []
f1_rtol_list= []


for pred_file, gt_file in zip(prediction_files, gt_files):


    pred_img= cv2.imread(os.path.join(OUTPUT_DIR_EXPTOSEED_COLOR_SAVED, pred_file), cv2.IMREAD_GRAYSCALE)
    # Apply same preprocessing as training dataset to match pixel alignment
    gt_img= preprocess_seed(os.path.join(SEED_FOLDER_EXPTOSIM_TEST_FIXED_NONFIXED_32X32, gt_file), img_length=32, img_width=32)


    pred_img_tensor= torch.from_numpy(pred_img)
    gt_img_tensor= torch.from_numpy(gt_img)

    pred_img_tensor_list.append(pred_img_tensor)
    gt_img_tensor_list.append(gt_img_tensor)

    precision, recall, f1_score = point_matching((pred_img_tensor),(gt_img_tensor), tolerance_radius=4, algorithm='hungarian')

    precision_rtol_list.append(precision)
    accuracy_rtol_list.append(recall)
    f1_rtol_list.append(f1_score)
    
    mse_value = mse_loss(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    dice_value = dice_loss(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    precision, recall, f1_score = precision_recall_f1(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    pixel_acc = pixel_accuracy(torch.from_numpy(pred_img), torch.from_numpy(gt_img))

    mse_loss_list.append(mse_value.item())
    dice_loss_list.append(dice_value.item())
    precision_recall_f1_list.append((precision, recall, f1_score))
    pixel_accuracy_list.append(pixel_acc)

    

In [5]:
# Experiments 

precision_avg = np.mean([pr[0] for pr in precision_recall_f1_list])
recall_avg = np.mean([pr[1] for pr in precision_recall_f1_list])
f1_avg = np.mean([pr[2] for pr in precision_recall_f1_list])

precision_std= np.std([pr[0] for pr in precision_recall_f1_list])
recall_std= np.std([pr[1] for pr in precision_recall_f1_list])
f1_std= np.std([pr[2] for pr in precision_recall_f1_list])  


print(f"MSE Mean: {np.mean(mse_loss_list)} Std: {np.std(mse_loss_list)}")
print(f"DICE loss Mean: {np.mean(dice_loss_list)} Std :{np.std(dice_loss_list)}")
print(f"Precision : {precision_avg:.3f} ± {precision_std:.3f}")
print(f"Recall : {recall_avg:.3f} ± {recall_std:.3f}")
print(f"F1 Score : {f1_avg:.3f} ± {f1_std:.3f}")
print(f"Pixel Accuracy average: {np.mean(pixel_accuracy_list):.3f} ± {np.std(pixel_accuracy_list):.3f}")



MSE Mean: 0.016416607481060608 Std: 0.011634062942219284
DICE loss Mean: 0.95033198595047 Std :0.10641436992145421
Precision : 0.062 ± 0.142
Recall : 0.047 ± 0.107
F1 Score : 0.050 ± 0.106
Pixel Accuracy average: 0.984 ± 0.012


In [6]:
# Experiments, tolerance point matching results (radius=4, Hungarian)

print(f"Tolerance Precision {np.mean(precision_rtol_list)} ± {np.std(precision_rtol_list)}")
print(f"Tolerance Recall {np.mean(accuracy_rtol_list)} ± {np.std(accuracy_rtol_list)}")
print(f"Tolerance F1 {np.mean(f1_rtol_list)} ± {np.std(f1_rtol_list)}")

from utils.evaluation_results import save_seed_metric_results
save_seed_metric_results(
    'Diffusion model\nExperiment→Seed', mse_loss_list, dice_loss_list,
    precision_rtol_list, accuracy_rtol_list, f1_rtol_list,
    prediction_files, gt_files, provenance=evaluation_provenance,
)


Tolerance Precision 0.7853832316950687 ± 0.20015782034375296
Tolerance Recall 0.6765500739315248 ± 0.28215166375017126
Tolerance F1 0.6834914194778084 ± 0.21697159585639406


{'comparison': 'Diffusion model\nExperiment→Seed',
 'n_images': 132,
 'mse_mean': 0.016416607481060608,
 'mse_std': 0.011634062942219284,
 'dice_mean': 0.95033198595047,
 'dice_std': 0.10641436992145421,
 'precision_mean': 0.7853832316950687,
 'precision_std': 0.20015782034375296,
 'recall_mean': 0.6765500739315248,
 'recall_std': 0.28215166375017126,
 'f1_mean': 0.6834914194778084,
 'f1_std': 0.21697159585639406}